# 多词元预测（Multi-Token Prediction MTP）

多数自回归模型每个位置只训练一个损失：即预测下一个词元。DeepSeek-V3 添加了额外的信息，继续多预测一个。不仅训练时的信号增加了，推理时也有了自然的草案。

## 问题描述

预测下一个词元是标准LLM的训练目标。所有的隐空间状态被监督用于准确预测一件事，下一个紧接着的词元，它带来的信号量出奇的弱。序列里大多数信号超过一个词元，结构、连贯性...。

MTP的核心理念是：如果每个隐空间状态监督用于一次预测多个词元呢。通过把多个独立的输出头之于骨架上，每个头预测一个位置的词元。并行、简单，但是各个头看到的信息是一样的，没有任何层级结构，预测也不遵循链式规则，所以不能被用于投机解码。

DeepSeek 把MTP设计成序列式的，模型从隐空间状态预测第一个词元，之后把这个词元的嵌入向量结合到隐空间状态上继续预测下一个词元，依此类推。嵌入矩阵和输出头可以直接复用主模型的，所以增加的参数量不算很多。

## 基本概念

### 序列MTP诀窍

将D个MTP模块附在主模型之上，第N个MTP预测之后第N个词元。

一个MTP模块包含：
- 一个Transformer块，包含自注意力和多层感知机。
- 一个投影矩阵，将上一深度的隐空间状态与词嵌入向量组合起来。
- 与主模型共享嵌入矩阵。
- 与主模型共享输出头。

训练的损失同时包含主词元的损失，还有各层MTP损失。

### 收益

加了MTP之后，预训练大概慢了10%，带来的收益则是：
1. 训练的信号更稠密。预训练后模型的基准变好了。
2. 推理期免费的投机解码。MTP自然生成了词元提案，第一个提案被接受的概率大概80%，换来1.8倍的推理效率提升。

### 对比EAGLE

|角度|EAGLE|MTP|
|---|---|---|
|训练时机|预训练之后|预训练过程中|
|参数|1-2层Transformer|1个Transformer块+1个投影块|
|接受率|0.88～0.92|0.80 at depth 1|
|加速以外的收益|无|更强的训练信号|

# 动手构建

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MTPModule(nn.Module):
    """DeepSeek-style sequential MTP block (toy).

    prev_hidden: (B, T, D)  previous depth / main-trunk hidden states
    next_embed:  (B, T, D)  embedding of the token this depth conditions on
                 (train: usually GT emb for the token at offset = depth)
    """

    def __init__(self, hidden_dim, ff_dim, num_heads=4):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        Norm = nn.RMSNorm if hasattr(nn, "RMSNorm") else nn.LayerNorm

        # Norm each stream, concat, project 2D -> D (paper-style; not sum)
        self.norm_h = Norm(hidden_dim)
        self.norm_e = Norm(hidden_dim)
        self.compress = nn.Linear(2 * hidden_dim, hidden_dim, bias=False)

        # Transformer block (Pre-Norm)
        self.attn_norm = Norm(hidden_dim)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.ffn_norm = Norm(hidden_dim)
        self.gate = nn.Linear(hidden_dim, ff_dim, bias=False)  # gate_proj
        self.up = nn.Linear(hidden_dim, ff_dim, bias=False)    # up_proj
        self.down = nn.Linear(ff_dim, hidden_dim, bias=False)  # down_proj

    def _attend(self, x):
        B, T, _ = x.shape
        H, Dh = self.num_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(Dh)
        # causal: still autoregressive along sequence positions
        causal = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.out_proj(out)

    def forward(self, prev_hidden, next_embed):
        # 1) fuse previous depth state with next-token embedding
        h = torch.cat([self.norm_h(prev_hidden), self.norm_e(next_embed)], dim=-1)
        h = self.compress(h)  # (B,T,2D) -> (B,T,D)

        # 2) one Transformer block
        h = h + self._attend(self.attn_norm(h))
        u = self.ffn_norm(h)
        h = h + self.down(F.silu(self.gate(u)) * self.up(u))
        return h


# smoke
B, T, D, FF = 2, 8, 64, 128
m = MTPModule(D, FF)
prev = torch.randn(B, T, D)
emb = torch.randn(B, T, D)
out = m(prev, emb)
assert out.shape == (B, T, D), out.shape
print("MTPModule ok", tuple(out.shape))


MTPModule ok (2, 8, 64)


In [ ]:
def ce_loss(logits, targets, ignore_index=-100):
    """logits: (B, T, V), targets: (B, T)"""
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=ignore_index,
    )


def mtp_total_loss(
    main_logits,          # (B, T, V) 主干：位置 t 预测 token_{t+1}
    mtp_logits_list,      # list of (B, T, V)；第 k 个深度预测更远的 token
    input_ids,            # (B, T)  当前输入 token ids
    lambda_mtp=0.3,       # MTP 总权重（DeepSeek 量级常用较小系数）
    ignore_index=-100,
):
    """
    对齐约定（常见写法）：
      input_ids[:, t] = token_t
      main:     用 hidden_t 预测 token_{t+1}  => target = input_ids[:, 1:]
      MTP depth k (k=1..D):
        用 h^{(k)}_t 预测 token_{t+1+k}
        => target 相对 main 再向右移 k

    序列尾部越界位置用 ignore_index 盖住。
    """
    B, T = input_ids.shape
    # ----- main next-token loss -----
    main_tgt = input_ids[:, 1:].contiguous()
    main_logits_ = main_logits[:, :-1, :].contiguous()
    loss_main = ce_loss(main_logits_, main_tgt, ignore_index)

    # ----- MTP losses -----
    loss_mtp = main_logits.new_zeros(())
    D = len(mtp_logits_list)
    for k, logits_k in enumerate(mtp_logits_list, start=1):
        # depth k predicts token_{t+1+k} at position t
        # usable positions: t = 0 .. T-2-k  => length T-1-k
        # logits_k[:, t] <-> target input_ids[:, t+1+k]
        shift = 1 + k
        if shift >= T:
            continue
        logits_ = logits_k[:, : T - shift, :].contiguous()
        tgt = input_ids[:, shift:].contiguous()
        loss_mtp = loss_mtp + ce_loss(logits_, tgt, ignore_index)

    if D > 0:
        loss_mtp = loss_mtp / D

    loss = loss_main + lambda_mtp * loss_mtp
    return {
        "loss": loss,
        "loss_main": loss_main.detach(),
        "loss_mtp": loss_mtp.detach(),
    }


# ---- 假数据 sanity check ----
V, B, T, Dhid = 100, 2, 8, 64
# fake shared lm_head
lm_head = nn.Linear(Dhid, V, bias=False)
emb = nn.Embedding(V, Dhid)

ids = torch.randint(0, V, (B, T))
# pretend trunk hidden states at each position
h0 = torch.randn(B, T, Dhid, requires_grad=True)
main_logits = lm_head(h0)

# depth-1 MTP: fuse h0 with emb(token_t) roughly — here emb of current ids
# (real training: roll/shift so next_embed aligns with the conditioned token)
mtp1 = MTPModule(Dhid, ff_dim=2 * Dhid)
h1 = mtp1(h0, emb(ids))
logits_1 = lm_head(h1)

out = mtp_total_loss(main_logits, [logits_1], ids, lambda_mtp=0.3)
out["loss"].backward()
print({k: float(v) if torch.is_tensor(v) else v for k, v in out.items()})
print("grad ok", h0.grad is not None and h0.grad.abs().mean().item() > 0)
